# Block 3: Audio Anomaly Detection mit Autoencoder
## PyTorch Deep Dive — Tensors, nn.Module, Training Loop, Hooks

**Szenario**: Sie sind Data Scientist bei einem Maschinenbauer. Ziel: Anomalien in Lüftergeräuschen erkennen — ohne gelabelte Fehlerdaten.  
**Ansatz**: Autoencoder, der nur auf normalen Daten trainiert wird und Anomalien durch hohen Rekonstruktionsfehler erkennt.

---

## 0. Setup & Datengenerierung

Diese Zelle ist **vorausgefüllt** — Datengenerierung ist nicht das Lernziel dieser Übung.  
Führen Sie sie aus und schauen Sie sich das Ergebnis an.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.decomposition import PCA

# Reproduzierbarkeit
torch.manual_seed(42)
np.random.seed(42)

# Device: GPU falls verfügbar, sonst CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {device}")

In [ ]:
def generate_spectrograms(n_normal=500, n_anomal=100, n_mels=64, n_frames=64):
    """Generiert synthetische Mel-Spektrogramm-ähnliche Daten.
    
    Normal: Regelmäßige harmonische Muster (wie ein gesunder Lüfter)
    Anomal: Harmonische Muster + Impulsstörungen + Frequenzverschiebungen
    
    Returns:
        normal: Tensor (n_normal, n_mels, n_frames)
        anomal: Tensor (n_anomal, n_mels, n_frames)
    """
    # Normal: base harmonics + small noise
    normal = torch.stack([
        torch.sin(torch.linspace(0, 4*np.pi, n_mels).unsqueeze(1).repeat(1, n_frames)
                  + torch.randn(1)*0.5) * 0.5 + torch.randn(n_mels, n_frames) * 0.1
        for _ in range(n_normal)
    ])

    # Anomal: harmonics + impulse noise + frequency shift
    anomal = torch.stack([
        torch.sin(torch.linspace(0, 4*np.pi, n_mels).unsqueeze(1).repeat(1, n_frames)
                  + torch.randn(1)*2.0) * 0.5  # verschobene Frequenzen
        + torch.randn(n_mels, n_frames) * 0.3   # mehr Rauschen
        + (torch.rand(n_mels, n_frames) > 0.95).float() * 2.0  # Impulsstörungen
        for _ in range(n_anomal)
    ])

    return normal, anomal


normal_data, anomal_data = generate_spectrograms()
print(f"Normale Spektrogramme:   {normal_data.shape}  (Samples × Mel-Bins × Frames)")
print(f"Anomale Spektrogramme:   {anomal_data.shape}")
print(f"Normaler Wertebereich:   [{normal_data.min():.2f}, {normal_data.max():.2f}]")
print(f"Anomaler Wertebereich:   [{anomal_data.min():.2f}, {anomal_data.max():.2f}]")

---
## Aufgabe A: PyTorch Basics (15 Min)

### A.1 Tensors und Device

PyTorch-Tensoren sind wie NumPy-Arrays — aber GPU-fähig und mit automatischer Gradientenberechnung.  
Erkunden Sie die grundlegenden Operationen.

In [ ]:
# DEIN CODE: Tensor-Basics
#
# 1. Erstelle einen Tensor der Form (3, 4) mit Zufallswerten (torch.randn)
# 2. Gib Shape, Dtype und Device aus
# 3. Schiebe ihn auf 'device' (.to(device))
# 4. Führe eine einfache Operation durch (z.B. Quadrierung oder Summe)


### A.2 Autograd — Automatische Gradientenberechnung

**Was ist Autograd?**  
Beim Training eines neuronalen Netzes braucht man den Gradienten des Loss bezüglich aller Parameter —  
um Gewichte in die richtige Richtung anzupassen. Das von Hand zu berechnen wäre bei Millionen von  
Parametern unmöglich.

PyTorch löst das mit **Autograd**: Jede Operation auf einem Tensor mit `requires_grad=True` wird in einem  
**Computation Graph** aufgezeichnet. Ein Aufruf von `.backward()` traversiert diesen Graphen rückwärts  
und berechnet automatisch alle Gradienten via Chain Rule.

Das Ergebnis steht in `.grad` des jeweiligen Tensors.

In [ ]:
# DEIN CODE: Autograd demonstrieren
#
# 1. Erstelle einen Tensor x mit requires_grad=True (z.B. x = torch.tensor([2.0, 3.0], requires_grad=True))
# 2. Berechne y = x**2 + 3*x
# 3. Rufe y.sum().backward() auf
# 4. Gib x.grad aus
# 5. Erkläre das Ergebnis: Warum hat x.grad diesen Wert? (dy/dx = 2x + 3 -- bei x=[2,3] also?)


*Ihre Erklärung*: ...

<details>
<summary><b>Hint A.2</b> (klicke zum Aufklappen)</summary>

**Was Sie erwarten sollten:**  
Der mathematische Gradient von `y = x² + 3x` ist `dy/dx = 2x + 3`.  
Für `x = [2.0, 3.0]` ergibt sich `dy/dx = [2*2+3, 2*3+3] = [7.0, 9.0]`.

**Relevante Doku:**  
- [Autograd Mechanics](https://pytorch.org/docs/stable/notes/autograd.html) — Wie der Computation Graph aufgebaut wird  
- [torch.Tensor.backward()](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html)  
- [A Gentle Introduction to torch.autograd](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)

</details>

In [ ]:
# Spektrogramme visualisieren (vorausgefüllt — zum Verständnis der Daten)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im0 = axes[0].imshow(normal_data[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Normales Spektrogramm')
axes[0].set_xlabel('Zeit-Frame')
axes[0].set_ylabel('Mel-Bin (Frequenz)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(anomal_data[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('Anomales Spektrogramm (Impulsstörungen + Frequenzshift)')
axes[1].set_xlabel('Zeit-Frame')
axes[1].set_ylabel('Mel-Bin (Frequenz)')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

---
## Aufgabe B: Daten & DataLoader (15 Min)

### Was ist ein Dataset und DataLoader?

`torch.utils.data.Dataset` kapselt Ihre Daten in einem standardisierten Interface:  
- `__len__()`: Wie viele Samples gibt es?  
- `__getitem__(idx)`: Gib Sample Nummer `idx` zurück.

`torch.utils.data.DataLoader` übernimmt dann Batching, Shuffling und optionales paralleles Laden.  
Dieses Interface ist der Standard in allen PyTorch-Projekten — von Bildklassifikation bis zu LLMs.

In [ ]:
# DEIN CODE: Custom Dataset und DataLoader
#
# 1. Implementiere die Klasse SpectrogramDataset(Dataset)
#    - __init__(self, data): speichere die Daten als self.data
#    - __len__(self): gib Anzahl der Samples zurück
#    - __getitem__(self, idx): gib self.data[idx] zurück
#
# 2. Erstelle einen DataLoader:
#    - Nur mit normalen Daten trainieren wir den Autoencoder!
#    - train_dataset = SpectrogramDataset(normal_data)
#    - train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
#
# 3. Verifiziere: Iteriere über einen Batch und prüfe die Shape.
#    Erwartete Shape: torch.Size([32, 64, 64])


<details>
<summary><b>Hint B</b> (klicke zum Aufklappen)</summary>

**Struktur einer Dataset-Klasse:**
```python
class MeinDataset(Dataset):
    def __init__(self, ...):
        ...
    def __len__(self):
        ...
    def __getitem__(self, idx):
        ...
```

**Relevante Doku:**  
- [Dataset & DataLoader Tutorial](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html)  
- [torch.utils.data.Dataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset)  
- [torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) — besonders `batch_size`, `shuffle`, `num_workers`

</details>

---
## Aufgabe C: Autoencoder bauen (20 Min)

### Das Autoencoder-Prinzip

```
Eingabe (64×64=4096)  →  Encoder  →  Bottleneck (32)  →  Decoder  →  Rekonstruktion (4096)
```

Der Bottleneck zwingt das Netz, nur die wesentlichen Eigenschaften zu lernen.  
Anomalien passen nicht in dieses komprimierte Schema → hoher Rekonstruktionsfehler.

**Architektur:**
- Encoder: `Linear(4096→512) → ReLU → Linear(512→128) → ReLU → Linear(128→32)`  
- Decoder: `Linear(32→128) → ReLU → Linear(128→512) → ReLU → Linear(512→4096)`

In [ ]:
# DEIN CODE: Autoencoder als nn.Module
#



<details>
<summary><b>Hint C</b> (klicke zum Aufklappen)</summary>

**Zwei wichtige Schritte in forward():**

1. **Flatten** vor dem Encoder: Das Netz erwartet 1D-Input, die Daten kommen als 2D-Matrizen.  
   Schauen Sie sich [torch.Tensor.view()](https://pytorch.org/docs/stable/tensor_view.html) an.

2. **Reshape** nach dem Decoder: Damit Loss und Visualisierung die ursprüngliche Form wiederhaben.

**Relevante Doku:**  
- [torch.nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html) — `__init__()` und `forward()` sind Pflicht  
- [torch.nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) — Layer hintereinander schalten  
- [torch.nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)  
- [torch.nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)  

**Parameteranzahl prüfen:**  
```python
sum(p.numel() for p in model.parameters())
```

</details>

---
## Aufgabe D: Training Loop (20 Min)

### Warum kein `.fit()`?

Scikit-Learn abstrahiert Training hinter `.fit()`. PyTorch tut das bewusst nicht.  
Der explizite Loop gibt Ihnen volle Kontrolle — und macht transparent, was passiert:

```
für jede Epoche:
    für jeden Batch:
        0. model.train()
        1. optimizer.zero_grad()      ← Gradienten auf null (PyTorch akkumuliert!)
        2. output = model(batch)      ← Forward Pass
        3. loss = criterion(output, batch)  ← Loss: MSE zwischen Eingabe und Rekonstruktion
        4. loss.backward()            ← Backpropagation: Gradienten berechnen
        5. optimizer.step()           ← Gewichte aktualisieren
```

**Warum `zero_grad()` vor jedem Schritt?**  
PyTorch akkumuliert Gradienten standardmäßig (nützlich für bestimmte Architekturen wie RNNs).  
Wenn Sie das nicht zurücksetzen, summieren sich alte und neue Gradienten → falsche Updates.

In [ ]:
# DEIN CODE: Training Loop
#
# Setup:



In [ ]:
# DEIN CODE: Loss-Kurve plotten
# plt.figure(figsize=(8, 4))
# plt.plot(train_losses)
# ...


In [ ]:
# MLflow Setup (optional — nur wenn MLflow installiert ist)
import os
try:
    import mlflow
    import mlflow.pytorch
    mlruns_path = os.path.join(os.getcwd(), 'mlruns')
    mlflow.set_tracking_uri(f"file://{mlruns_path}")
    mlflow.set_experiment('Block3-Audio-Anomaly-Detection')
    print(f"MLflow konfiguriert. Tracking: {mlruns_path}")
    print(f"UI starten mit: mlflow ui --backend-store-uri file://{mlruns_path}")
    MLFLOW_AVAILABLE = True
except ImportError:
    print("MLflow nicht installiert — Überspringen Sie die MLflow-Aufgaben.")
    print("Installation: pip install mlflow")
    MLFLOW_AVAILABLE = False

# DEIN CODE: Training in MLflow loggen (nur wenn MLFLOW_AVAILABLE)
# with mlflow.start_run(run_name="autoencoder-baseline"):
#     mlflow.log_param("latent_dim", 32)
#     mlflow.log_param("lr", 1e-3)
#     mlflow.log_param("n_epochs", 50)
#     mlflow.log_metric("final_train_loss", train_losses[-1])
#     # Loss-Plot als Artefakt speichern...

<details>
<summary><b>Hint D</b> (klicke zum Aufklappen)</summary>


**`model.train()` vs. `model.eval()`:**  
Manche Layer (Dropout, BatchNorm) verhalten sich beim Training anders als bei Inferenz.  
`model.train()` schaltet Training-Modus ein, `model.eval()` Inferenz-Modus.  
Gute Praxis: immer explizit setzen.

**Relevante Doku:**  
- [Training Loop Tutorial](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)  
- [torch.optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)  
- [nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)  
- [MLflow Tracking Quickstart](https://mlflow.org/docs/latest/getting-started/intro-quickstart/index.html)

</details>

---
## Aufgabe E: Anomalie-Detektion (15 Min)

### Reconstruction Error als Anomalie-Score

Der trainierte Autoencoder kennt nur normale Spektrogramme.  
**Hypothese**: Anomale Spektrogramme → höherer Rekonstruktionsfehler.

Berechnen Sie für jeden Datenpunkt den **per-Sample MSE** zwischen Eingabe und Rekonstruktion.  
→ Normales Sample: kleiner MSE  
→ Anomales Sample: großer MSE  

**Schwellenwert τ** trennt normale von anomalen Samples:  
τ = μ_normal + 2 × σ_normal  (aus dem Reconstruction Error auf normalen Testdaten)

In [ ]:
# DEIN CODE: Reconstruction Error berechnen
#
# 1. Erstelle einen Test-Split: z.B. 400 normal für Training, 100 normal für Test + alle 100 anomal
#    (Hinweis: Die normalen Testdaten sollten das Modell nie gesehen haben)
#    normal_test = normal_data[400:]  # letzte 100 normalen Samples als Testset
#
# 2. Berechne per-Sample MSE:
#    - torch.no_grad() verwenden für Inferenz (spart Speicher!)
#    - model.eval()
#    - Für jeden Datenpunkt: MSE zwischen Eingabe und Rekonstruktion
#    - Ergebnis: zwei Listen/Tensoren (normal_errors, anomal_errors)
#
# Tipp für per-Sample MSE (nicht über alle Samples mitteln):
# error = ((x - x_hat)**2).mean(dim=[1, 2])  # mittelt über Mel-Bins und Frames, nicht über Batch


In [ ]:
# DEIN CODE: Fehlerverteilungen visualisieren
#
# Plotte beide Verteilungen als überlagertes Histogramm (alpha=0.6, bins=40)
# Beschrifte mit Legende (Normal / Anomal) und zeichne den Schwellenwert τ ein


In [ ]:
# DEIN CODE: Schwellenwert setzen und Metriken berechnen
#
# tau = normal_errors.mean() + 2 * normal_errors.std()
# print(f"Schwellenwert τ: {tau:.4f}")
#
# Klassifiziere: Fehler > tau → Anomalie (1), sonst Normal (0)
# Berechne Precision, Recall, F1 für die Anomalie-Klasse


**Verbindung zu Block 2 — Kosten-Asymmetrie:**  

*Wenn Sie τ senken (strengerer Detektor):* ...

*Wenn Sie τ erhöhen (entspannter Detektor):* ...

*In welchem Fall würden Sie τ senken?* ...

<details>
<summary><b>Hint E</b> (klicke zum Aufklappen)</summary>

**torch.no_grad() für Inferenz:**  
```python
with torch.no_grad():
    reconstructed = model(data.to(device))
```
Dies deaktiviert den Computation Graph → kein Speicher für Gradienten → schneller.

**Per-Sample MSE (nicht Batch-Mittelwert):**  
`nn.MSELoss()` mittelt über alle Dimensionen. Sie wollen den Fehler **pro Sample**:  
```python
errors = ((original - reconstructed) ** 2).mean(dim=[1, 2])
```

**Relevante Doku:**  
- [torch.no_grad()](https://pytorch.org/docs/stable/generated/torch.no_grad.html)  
- [sklearn.metrics.precision_score, recall_score, f1_score](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics)

</details>

---
## Aufgabe F: Hooks & Latent Space (15 Min)

### Was sind Forward Hooks?

Ein **Forward Hook** ist eine Callback-Funktion, die automatisch ausgeführt wird,  
wenn ein Modul seinen Forward Pass abschließt. Sie sehen Input und Output — ohne das Modell zu verändern.

```python
latent_activations = []

def mein_hook(module, input, output):
    latent_activations.append(output.detach().cpu())

handle = model.encoder[-1].register_forward_hook(mein_hook)
# ... Inference ...
handle.remove()  # Hook wieder entfernen!
```

**Warum ist das wichtig?**  
Hooks ermöglichen nicht-invasive Inspektion — das ist die technische Basis für **Mechanistic Interpretability**.  
In Block 4/5 nutzen wir dasselbe Prinzip, um Attention Heads in GPT-2 zu inspizieren.

In [ ]:
# DEIN CODE: Forward Hook registrieren und Latent Vectors sammeln
#
# 1. Erstelle eine leere Liste latent_activations = []
# 2. Definiere einen Hook, der output.detach().cpu() in die Liste anhängt
# 3. Registriere den Hook auf model.encoder[-1] (letzter Layer des Encoders = Bottleneck)
# 4. Schicke ALLE Daten (normal + anomal) durch das Modell
# 5. Entferne den Hook mit handle.remove()
# 6. Konkateniere alle latent_activations zu einem Tensor
#    Erwartete Shape: (600, 32) — 500 normal + 100 anomal


In [ ]:
# DEIN CODE: Latent Space visualisieren (PCA oder t-SNE)
#
# 1. Labels erstellen: [0]*500 + [1]*100 (0=normal, 1=anomal)
# 2. PCA auf 2D reduzieren (schnell) ODER t-SNE (langsameer aber oft schöner)
# 3. Scatter Plot: normale Samples in einer Farbe, anomale in einer anderen
# 4. Frage im Notebook beantworten: Sind die Klassen separierbar?


**Beobachtung:**  
*Sind normale und anomale Samples im Latent Space separierbar?*  

...

*Was bedeutet das für die Qualität des gelernten Bottlenecks?*  

...

<details>
<summary><b>Hint F</b> (klicke zum Aufklappen)</summary>

**Hook-Muster:**
```python
collected = []
def hook_fn(module, input, output):
    collected.append(output.detach().cpu())
handle = irgendein_modul.register_forward_hook(hook_fn)
# ... inference ...
handle.remove()
result = torch.cat(collected, dim=0)
```

**Relevante Doku:**  
- [torch.nn.Module.register_forward_hook()](https://pytorch.org/docs/stable/generated/torch.nn.Module.register_forward_hook.html)  
- [sklearn.decomposition.PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)  
- [sklearn.manifold.TSNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) — `n_components=2, perplexity=30`

**Tipp**: Für t-SNE müssen Sie den Tensor zuerst in NumPy konvertieren: `.numpy()`

</details>

---
## Aufgabe G: Reflexion (10 Min)

Beantworten Sie die folgenden Fragen (je 3-4 Sätze). Keine Code-Antworten — es geht um Urteilsvermögen.

**Frage 1**: Der Schwellenwert τ = μ + 2σ ist eine Heuristik. Was passiert konkret, wenn Sie τ senken? Und wenn Sie τ erhöhen? In welchem Anwendungsfall würden Sie τ senken — und warum hängt das mit der Kosten-Asymmetrie aus Block 2 zusammen?

*Ihre Antwort*: ...

**Frage 2**: Ein Kollege schlägt vor, den Autoencoder auch auf einem kleinen Anteil anomaler Daten zu trainieren, "damit er mehr sieht". Warum ist das ein Problem? Unter welchen Bedingungen würde der Autoencoder auch Anomalien gut rekonstruieren — selbst wenn er nur auf Normaldaten trainiert wurde?

*Ihre Antwort*: ...